In [ ]:
import os
import shutil
import xml.etree.ElementTree as ET
import zarr
import numpy as np

def copy_zarr_with_attributes(xml_file_path, new_base_location):
    """
    Parses a SpimData XML file, copies associated Zarr tiles to a new
    location, and adds size and position attributes to each copied tile.

    Args:
        xml_file_path (str): The full path to the XML metadata file.
        new_base_location (str): The path to the new directory where Zarr
                                 tiles will be copied.
    """
    print(f"?? Parsing XML file: {xml_file_path}")

    try:
        tree = ET.parse(xml_file_path)
        root = tree.getroot()
    except ET.ParseError as e:
        print(f"Error parsing XML file: {e}")
        return
    except FileNotFoundError:
        print(f"Error: XML file not found at {xml_file_path}")
        return

    # 1. Get the absolute base path of the original Zarr files from the XML
    original_base_path = root.find('.//ImageLoader/zarr').text
    if not original_base_path:
        print("Error: Could not find the base Zarr path in the XML file.")
        return
    
    print(f"?? Original data location: {original_base_path}")

    # 2. Create the destination directory if it doesn't exist
    os.makedirs(new_base_location, exist_ok=True)
    print(f"?? Destination directory: {new_base_location}")

    # 3. Extract tile metadata (size and position)
    tile_metadata = {}

    # Get size for each tile from <ViewSetups>
    for view_setup in root.findall('.//ViewSetup'):
        setup_id = view_setup.find('id').text
        tile_name = view_setup.find('name').text
        size_str = view_setup.find('size').text
        
        # Convert size string "width height depth" to a list of integers
        size = [int(s) for s in size_str.split()]
        
        tile_metadata[setup_id] = {'name': tile_name, 'size': size}

    # Get position for each tile from <ViewRegistrations>
    for view_reg in root.findall('.//ViewRegistration'):
        setup_id = view_reg.get('setup')
        if setup_id in tile_metadata:
            affine_str = view_reg.find('.//affine').text
            affine_values = [float(v) for v in affine_str.split()]
            
            # The translation vector (tx, ty, tz) is at indices 3, 7, and 11
            # in the 3x4 affine matrix flattened row-wise.
            position = [affine_values[3], affine_values[7], affine_values[11]]
            tile_metadata[setup_id]['position'] = position

    if not tile_metadata:
        print("Warning: No tile information was found in the XML file.")
        return

    # 4. Copy each Zarr tile and add the new attributes
    print("\n?? Starting tile processing...")
    for setup_id, data in tile_metadata.items():
        tile_name = data['name']
        source_path = os.path.join(original_base_path, tile_name)
        dest_path = os.path.join(new_base_location, tile_name)

        print(f"\nProcessing Tile ID {setup_id}: {tile_name}")

        # Check if the source Zarr directory exists
        if not os.path.exists(source_path):
            print(f"  - ?? Source tile not found: {source_path}. Skipping.")
            continue

        # Copy the Zarr directory tree
        print(f"  - Copying from {source_path} to {dest_path}")
        try:
            shutil.copytree(source_path, dest_path, dirs_exist_ok=True)
        except Exception as e:
            print(f"  - ? Error copying directory: {e}")
            continue

        # Add attributes to the newly copied Zarr file
        print("  - Adding attributes...")
        try:
            z_store = zarr.open(dest_path, mode='r+')
            z_store.attrs['tile_size'] = data['size']
            z_store.attrs['position'] = data['position']
            print(f"  - ? Attributes added: size={data['size']}, position={data['position']}")
        except Exception as e:
            print(f"  - ? Error opening or writing attributes to {dest_path}: {e}")

    print("\n?? All tiles processed successfully!")


if __name__ == '__main__':
    # --- PLEASE UPDATE THESE PATHS ---
    # Path to your XML file
    xml_input_path = "/path/to/your/dataset.xml" 
    
    # Path to the new directory where you want to store the modified Zarr files
    new_zarr_output_directory = "/path/to/your/new_zarr_location"
    
    # --- Example Usage ---
    # Before running, make sure the dummy data setup below matches your real paths
    # or point the paths above to your actual data.
    
    # NOTE: To run this script, replace the placeholder paths above with the
    # actual paths to your XML file and desired output directory.
    if xml_input_path == "/path/to/your/dataset.xml":
        print("?? Welcome! Please update the 'xml_input_path' and 'new_zarr_output_directory' variables in the script.")
    else:
        copy_zarr_with_attributes(xml_input_path, new_zarr_output_directory)